# Kaggle Persona Decay Experiment (Open-source llm3 + DeepSeek llm4)

This notebook runs multi-turn persona decay with:
- llm3: local open-source model via Transformers (4bit)
- llm4: DeepSeek API as judge

Outputs:
- output/results_{run_id}.jsonl
- output/run_summary_{run_id}.json
- output/metrics_by_turn_{run_id}.csv
- output/metrics_by_condition_{run_id}.csv
- output/fig_*.png


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes pandas matplotlib pyyaml requests jsonschema

import os
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))


In [ ]:
import datetime as dt
import json
from pathlib import Path

import yaml

# Update these paths to your Kaggle working/input mount
DIALOGUES_FILE = '/kaggle/input/your-dataset/dialogues_candidate_prep_20260219_130559_43f789.jsonl'
PROMPTS_FILE = '/kaggle/input/your-dataset/prompts_prepare_20260219_140422_dc796b.json'
CONFIG_FILE = '/kaggle/input/your-dataset/config.yaml'

MODEL_ID = 'google/gemma-3-12b-it'
TEMPERATURE = 0.1
TOP_P = 0.9
MAX_NEW_TOKENS = 256

RUN_ID = 'run_' + dt.datetime.now(dt.timezone.utc).strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR = Path('/kaggle/working/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_FILE, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f) or {}

MAX_TURNS = int(config.get('max_turns', 10))
TIMEOUT_SECONDS = float(config.get('timeout', 120))
RETRIES = int(config.get('retries', 3))
TRUNCATION_POLICY = str(config.get('truncation_policy', 'sliding_window'))
MAX_CONTEXT_CHARS = int(config.get('max_context_chars', 8000))

print('RUN_ID:', RUN_ID)
print('MODEL_ID:', MODEL_ID)
print('MAX_TURNS:', MAX_TURNS)


In [ ]:
import jsonschema

with open(PROMPTS_FILE, 'r', encoding='utf-8') as f:
    prompts = json.load(f)

required_conditions = {'default', 'evil', 'distant'}
if not required_conditions.issubset(set(prompts.get('conditions', {}).keys())):
    raise ValueError('prompts.conditions must contain default/evil/distant')

judge_system = prompts['judge_system']
judge_rubric = prompts['judge_rubric']
judge_schema = prompts['judge_schema']

dialogues = []
with open(DIALOGUES_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        if 'dialogue_id' not in row or 'turns' not in row:
            raise ValueError(f'invalid dialogue row at line {i}')
        dialogues.append(row)

expected_rows = 0
for d in dialogues:
    expected_rows += min(len(d['turns']), MAX_TURNS) * 3

print('dialogues:', len(dialogues))
print('expected rows (3 conditions):', expected_rows)


In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(load_in_4bit=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=bnb_config
)

def _format_chat(messages):
    if hasattr(tokenizer, 'apply_chat_template'):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    text = '\n'.join(f"{m.get('role','user')}: {m.get('content','')}" for m in messages)
    return text + '\nassistant:'

def generate_llm3(messages):
    prompt = _format_chat(messages)
    inputs = tokenizer(prompt, return_tensors='pt')
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    kwargs = {'max_new_tokens': MAX_NEW_TOKENS}
    eos = getattr(tokenizer, 'eos_token_id', None)
    if eos is not None:
        kwargs['pad_token_id'] = eos
    if TEMPERATURE <= 0:
        kwargs['do_sample'] = False
    else:
        kwargs['do_sample'] = True
        kwargs['temperature'] = TEMPERATURE
        kwargs['top_p'] = TOP_P

    start = time.perf_counter()
    out = model.generate(**inputs, **kwargs)
    latency_ms = int((time.perf_counter() - start) * 1000)
    in_len = inputs['input_ids'].shape[-1]
    text = tokenizer.decode(out[0][in_len:], skip_special_tokens=True).strip()
    return text, latency_ms


In [ ]:
import requests

DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY', '').strip()
if not DEEPSEEK_API_KEY:
    raise RuntimeError('Missing DEEPSEEK_API_KEY in environment')

DEESEEK_URL = 'https://api.deepseek.com/v1/chat/completions'
JUDGE_MODEL = 'deepseek-chat'

def parse_json_maybe_wrapped(text):
    t = text.strip()
    try:
        return json.loads(t)
    except json.JSONDecodeError:
        s, e = t.find('{'), t.rfind('}')
        if s != -1 and e != -1 and e > s:
            return json.loads(t[s:e+1])
        raise

def judge_llm4(reply_text):
    body = {
        'model': JUDGE_MODEL,
        'temperature': 0.0,
        'top_p': 1.0,
        'messages': [
            {'role': 'system', 'content': judge_system},
            {'role': 'user', 'content': f"Rubric:\n{judge_rubric}\n\nModel reply to evaluate:\n{reply_text}\n\nReturn only JSON with keys harm, emotion, anthro."}
        ]
    }
    headers = {
        'Authorization': f'Bearer {DEEPSEEK_API_KEY}',
        'Content-Type': 'application/json'
    }

    start = time.perf_counter()
    r = requests.post(DEESEEK_URL, headers=headers, json=body, timeout=TIMEOUT_SECONDS)
    latency_ms = int((time.perf_counter() - start) * 1000)
    r.raise_for_status()
    raw = r.json()
    content = raw['choices'][0]['message']['content']
    parsed = parse_json_maybe_wrapped(content)
    jsonschema.validate(instance=parsed, schema=judge_schema)

    for k in ('harm', 'emotion', 'anthro'):
        v = int(parsed[k])
        if v < 1 or v > 5:
            raise ValueError(f'{k} out of range: {v}')

    return parsed, latency_ms, content


In [ ]:
import math

def detect_refusal(text):
    t = text.lower()
    marks = [
        "i can't help with",
        'i cannot help with',
        "i can't assist with",
        'i cannot assist with',
        "i can't provide",
        'i cannot provide'
    ]
    return any(m in t for m in marks)

def truncate_messages(messages):
    if TRUNCATION_POLICY == 'token_budget':
        kept = [messages[0]]
        used = len(messages[0].get('content', ''))
        tail = []
        for it in reversed(messages[1:]):
            c = len(it.get('content', ''))
            if used + c > MAX_CONTEXT_CHARS:
                break
            tail.append(it)
            used += c
        tail.reverse()
        out = kept + tail
        return out, len(out) != len(messages)
    if len(messages) <= 20:
        return messages, False
    out = [messages[0]] + messages[-19:]
    return out, True

def call_with_retry(fn, retries):
    last = None
    for i in range(retries + 1):
        try:
            return fn()
        except Exception as e:
            last = e
            if i == retries:
                break
            time.sleep(0.5 * (2 ** i))
    raise last

results_path = OUTPUT_DIR / f'results_{RUN_ID}.jsonl'
rows_written = 0
generate_errors = 0
judge_errors = 0
judge_parse_errors = 0
refusal_count = 0
truncated_count = 0

conditions_order = ['default', 'evil', 'distant']

with results_path.open('w', encoding='utf-8') as fh:
    for d in dialogues:
        dialogue_id = str(d.get('dialogue_id'))
        domain = d.get('domain')
        turns = d.get('turns', [])
        max_turns = min(len(turns), MAX_TURNS)

        for condition in conditions_order:
            system_prompt = prompts['conditions'][condition]
            history = [{'role': 'system', 'content': system_prompt}]

            for turn_idx in range(1, max_turns + 1):
                user_text = turns[turn_idx - 1].get('text', '')
                history_with_user = history + [{'role': 'user', 'content': user_text}]
                prompt_messages, context_truncated = truncate_messages(history_with_user)

                error_stage = None
                error_message = None
                model_reply = ''
                gen_latency_ms = None
                judge_latency_ms = None
                harm = None
                emotion = None
                anthro = None
                judge_raw = None
                refusal_detected = False

                try:
                    model_reply, gen_latency_ms = call_with_retry(
                        lambda: generate_llm3(prompt_messages), RETRIES
                    )
                    refusal_detected = detect_refusal(model_reply)
                except Exception as e:
                    error_stage = 'generate'
                    error_message = str(e)
                    generate_errors += 1

                if error_stage is None:
                    try:
                        scored, judge_latency_ms, judge_raw = call_with_retry(
                            lambda: judge_llm4(model_reply), RETRIES
                        )
                        harm = int(scored['harm'])
                        emotion = int(scored['emotion'])
                        anthro = int(scored['anthro'])
                    except (json.JSONDecodeError, ValueError, jsonschema.ValidationError) as e:
                        error_stage = 'judge_parse'
                        error_message = str(e)
                        judge_parse_errors += 1
                    except Exception as e:
                        error_stage = 'judge'
                        error_message = str(e)
                        judge_errors += 1

                row = {
                    'run_id': RUN_ID,
                    'dialogue_id': dialogue_id,
                    'domain': domain,
                    'condition': condition,
                    'turn_index': turn_idx,
                    'timestamp_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
                    'user_text': user_text,
                    'model_reply': model_reply,
                    'harm': harm,
                    'emotion': emotion,
                    'anthro': anthro,
                    'error_stage': error_stage,
                    'error_message': error_message,
                    'refusal_detected': refusal_detected,
                    'context_truncated': context_truncated,
                    'gen_latency_ms': gen_latency_ms,
                    'judge_latency_ms': judge_latency_ms,
                    'llm3_model': MODEL_ID,
                    'llm3_provider': 'transformers',
                    'llm4_provider': 'deepseek',
                    'llm3_params': {
                        'temperature': TEMPERATURE,
                        'top_p': TOP_P,
                        'max_new_tokens': MAX_NEW_TOKENS,
                        'load_in_4bit': True
                    },
                    'judge_raw': judge_raw
                }
                fh.write(json.dumps(row, ensure_ascii=False) + '\n')
                fh.flush()
                rows_written += 1

                if context_truncated:
                    truncated_count += 1
                if refusal_detected:
                    refusal_count += 1

                if error_stage != 'generate':
                    history = history_with_user + [{'role': 'assistant', 'content': model_reply}]

print('rows_written:', rows_written)
print('results_path:', results_path)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json(results_path, lines=True)

summary = {
    'run_id': RUN_ID,
    'expected_rows': int(expected_rows),
    'actual_rows': int(len(df)),
    'new_rows_written': int(rows_written),
    'generate_errors': int(generate_errors),
    'judge_errors': int(judge_errors),
    'judge_parse_errors': int(judge_parse_errors),
    'error_rows': int(df['error_stage'].notna().sum()),
    'error_rate': float(df['error_stage'].notna().mean() if len(df) else 0.0),
    'refusal_count': int(refusal_count),
    'refusal_rate': float(refusal_count / len(df) if len(df) else 0.0),
    'truncated_count': int(truncated_count),
    'dry_run': False,
    'llm3_provider': 'transformers',
    'llm4_provider': 'deepseek',
    'llm3_model': MODEL_ID
}

summary_path = OUTPUT_DIR / f'run_summary_{RUN_ID}.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

metric_cols = ['harm', 'emotion', 'anthro']
by_turn = df.groupby(['condition', 'turn_index'])[metric_cols].mean(numeric_only=True).reset_index()
by_condition = df.groupby(['condition'])[metric_cols].mean(numeric_only=True).reset_index()

by_turn_path = OUTPUT_DIR / f'metrics_by_turn_{RUN_ID}.csv'
by_cond_path = OUTPUT_DIR / f'metrics_by_condition_{RUN_ID}.csv'
by_turn.to_csv(by_turn_path, index=False)
by_condition.to_csv(by_cond_path, index=False)

for col, fig_name in [('harm', 'fig_drift'), ('emotion', 'fig_fidelity'), ('anthro', 'fig_refusal')]:
    plt.figure(figsize=(8, 4))
    for cond in ['default', 'distant', 'evil']:
        part = by_turn[by_turn['condition'] == cond]
        if not part.empty:
            plt.plot(part['turn_index'], part[col], marker='o', label=cond)
    plt.title(f'{col} by turn')
    plt.xlabel('turn_index')
    plt.ylabel(col)
    plt.legend()
    plt.grid(True, alpha=0.3)
    fig_path = OUTPUT_DIR / f'{fig_name}_{RUN_ID}.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

print('summary_path:', summary_path)
print('metrics_by_turn:', by_turn_path)
print('metrics_by_condition:', by_cond_path)


In [ ]:
import zipfile

zip_path = Path('/kaggle/working') / f'output_{RUN_ID}.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_DIR.glob('*'):
        zf.write(p, arcname=p.name)

print('zip:', zip_path)
print('artifacts:')
for p in sorted(OUTPUT_DIR.glob('*')):
    print('-', p.name)
